# 04 · Retrieve, prune, rerank, with label cards

The funnel from MATCH (arXiv 2610.25948): find candidate labels across the whole taxonomy by similarity, drop weak
branches using the tiers with no model involved, then have a model choose among what is left. Label cards are from SLKR
(arXiv 2610.16182): a model writes a definition for each label once, offline, from the taxonomy alone.

**Reduced forms.** One view of a text rather than MATCH's eight, one model rather than a tiered one, and no pruning by tier,
because both a short second view and pruning lost correct labels on the development split, in exploratory work before this repository. On HumSet, with
only 75 nodes, nothing is narrowed: the funnel offers every label, so it tests label cards only.

**An addition.** Notebook 02's candidate recall shows retrieval alone misses most correct labels on EuroVoc, which caps any funnel. So
`funnel_hybrid`, on MultiEURLEX only, also offers the trained classifier's strongest labels: 50 from each source. This is not
from either paper.

**What would count as what, written before running.**
- A funnel cannot choose a label it never retrieved, so candidate recall is an upper limit on its recall, reported alongside.
- Expected from SLKR: label cards help, more for a small taxonomy with bare names.
- Expected from MATCH: the funnel beats the trained model on exact labels only if retrieval finds enough. Open.
- Differences count only where the 95% paired bootstrap interval excludes zero.


In [1]:
import pandas as pd

from tiered_taxonomy import experiment as ex
from tiered_taxonomy import runs
from tiered_taxonomy.config import data_root, load_config

root = data_root()
config = load_config()
seed = config["seed"]
samples = config["samples"]


def sample_for(data):
    if data.name == ex.EURLEX:
        settings = samples["eurlex"]
        return ex.sample_eurlex(
            data,
            settings["normal_documents"],
            settings["long_documents"],
            settings["long_threshold_words"],
            seed,
        )
    return ex.sample_humset(data, samples["humset"]["per_language"], seed)


def table(results):
    rows = []
    for name, result in results.items():
        scores = result["scores"]
        row = {
            "method": name,
            "labels per item": scores["labels_per_item"],
            "exact F1": round(scores["exact"]["f1"], 3),
            "exact precision": round(scores["exact"]["precision"], 3),
            "exact recall": round(scores["exact"]["recall"], 3),
        }
        row.update({f"tier {tier} F1": round(value["f1"], 3) for tier, value in scores["tiers"].items()})
        row.update({f"{band} recall": value["recall"] for band, value in scores["rarity"].items()})
        if "run" in result:
            run = result["run"]
            row.update(
                {
                    "calls per item": round(run["calls"] / run["items"], 1),
                    "cost per 1000 items ($)": round(1000 * run["cost_usd"] / run["items"], 2),
                    "malformed": run["malformed"],
                    "invalid": run["invalid"],
                }
            )
        rows.append(row)
    return pd.DataFrame(rows)

/Users/matthewharris/git/nvidia-autoontology-demo/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
eurlex = ex.load_eurlex(root)
humset = ex.load_humset(root)
benches = {data.name: runs.make_bench(data, root, config) for data in (eurlex, humset)}
sets = {data.name: data for data in (eurlex, humset)}
items = {name: sample_for(data) for name, data in sets.items()}
for name, bench in benches.items():
    runs.load_cards(bench)
    runs.embed_inputs(bench, items[name])
used = ex.project_labels([*humset.train, *humset.test], humset.taxonomy)
results = {}

In [3]:
from tiered_taxonomy.metrics import candidate_recall, micro_f1, paired_difference

recall_grid = {}
for name in sets:
    bench = benches[name]
    grid = config["funnel"]["candidate_grid"]
    recall_grid[name] = {
        "names": runs.candidate_grid(bench, items[name], bench.index, grid),
        "cards": runs.candidate_grid(bench, items[name], bench.card_index, grid),
    }
    methods = ["funnel_names", "funnel_cards"] + (["funnel_hybrid"] if name == ex.EURLEX else [])
    group = {method: runs.run_and_score(bench, method, items[name]) for method in methods}
    results[name] = group
pd.DataFrame(
    {
        (name, kind): {k: row["all"] for k, row in rows.items()}
        for name, kinds in recall_grid.items()
        for kind, rows in kinds.items()
    }
).T

5      10      20      50     100     200
eurlex names  0.0910  0.1424  0.1992  0.3055  0.4055  0.5229
       cards  0.0877  0.1286  0.1831  0.2733  0.3452  0.4439
humset names  0.1649  0.2681  0.4497  0.8237  1.0000  1.0000
       cards  0.2800  0.4027  0.5693  0.8813  1.0000  1.0000

## How much of the gold label set survives the funnel's first steps?

In [4]:
from tiered_taxonomy.methods.llm_methods import funnel_candidates

reached = {}
for name in sets:
    bench = benches[name]
    sample = items[name]
    gold = ex.gold_exact(sample)
    settings = bench.funnel_settings()
    for kind, index in (("names", bench.index), ("cards", bench.card_index)):
        candidates = [
            set(funnel_candidates(ex.input_text(item, bench.words), index, bench.store, settings))
            for item in sample
        ]
        reached[f"{name} {kind}"] = round(candidate_recall(gold, candidates), 4)
reached

{'eurlex names': 0.4055,
 'eurlex cards': 0.3452,
 'humset names': 1.0,
 'humset cards': 1.0}

## Where do correct labels get lost?

The share of correct labels offered to the model, against the share it kept. The trained classifier's own top 6 is alongside
for comparison, since the hybrid is offered that classifier's candidates.

In [5]:
import json

from tiered_taxonomy.methods.llm_methods import hybrid_candidates
from tiered_taxonomy.metrics import top_k

sample = items[ex.EURLEX]
gold = ex.gold_exact(sample)
bench = benches[ex.EURLEX]
ranked = json.loads(runs.trained_ranked_path(root, ex.EURLEX).read_text(encoding="utf-8"))
each = config["funnel"]["hybrid_candidates_each"]
settings = bench.funnel_settings()
offered = [
    set(
        hybrid_candidates(
            ranked[item.id],
            funnel_candidates(ex.input_text(item, bench.words), bench.card_index, bench.store, settings),
            each,
        )
    )
    for item in sample
]
stages = {
    "hybrid: offered to the model": round(candidate_recall(gold, offered), 4),
    "hybrid: kept by the model": results[ex.EURLEX]["funnel_hybrid"]["scores"]["exact"]["recall"],
    "hybrid: candidates per law": round(sum(len(labels) for labels in offered) / len(offered), 1),
    "trained: top 50 alone": round(
        candidate_recall(gold, [top_k(ranked[item.id], 50) for item in sample]), 4
    ),
    "trained: its top 6": results[ex.EURLEX]["trained"]["scores"]["exact"]["recall"]
    if "trained" in results[ex.EURLEX]
    else None,
    "funnel with cards: offered": reached["eurlex cards"],
    "funnel with cards: kept": results[ex.EURLEX]["funnel_cards"]["scores"]["exact"]["recall"],
    "funnel with names: offered": reached["eurlex names"],
    "funnel with names: kept": results[ex.EURLEX]["funnel_names"]["scores"]["exact"]["recall"],
}
trained_labels = ex.load_predictions(root, ex.EURLEX, "trained", [item.id for item in sample])
stages["trained: its top 6"] = ex.evaluate(eurlex, sample, trained_labels)["exact"]["recall"]
stages

{'hybrid: offered to the model': 0.7456,
 'hybrid: kept by the model': 0.2030171311684991,
 'hybrid: candidates per law': 95.6,
 'trained: top 50 alone': 0.687,
 'trained: its top 6': 0.40322168243416007,
 'funnel with cards: offered': 0.3452,
 'funnel with cards: kept': 0.13040143185885963,
 'funnel with names: offered': 0.4055,
 'funnel with names: kept': 0.15034518026080287}

## MultiEURLEX

In [6]:
table(results[ex.EURLEX])

,method,labels per item,exact F1,exact precision,exact recall,tier 1 F1,tier 2 F1,tier 3 F1,common recall,rare recall,unseen recall,calls per item,cost per 1000 items ($),malformed,invalid
0,funnel_names,3.88,0.188,0.252,0.150,0.428,0.319,0.274,0.1484,0.1505,0.2083,1.0,0.46,8,0
1,funnel_cards,3.76,0.165,0.226,0.130,0.414,0.307,0.260,0.1248,0.1774,0.2250,1.0,0.60,4,0
2,funnel_hybrid,4.36,0.243,0.303,0.203,0.536,0.421,0.364,0.2061,0.1452,0.2000,1.0,0.61,7,4


## HumSet

In [7]:
table(results[ex.HUMSET])

,method,labels per item,exact F1,exact precision,exact recall,tier 1 F1,tier 2 F1,common recall,rare recall,calls per item,cost per 1000 items ($),malformed,invalid
0,funnel_names,2.52,0.365,0.353,0.377,0.439,0.261,0.3770,0.4032,1.0,0.18,0,5
1,funnel_cards,1.75,0.395,0.463,0.344,0.445,0.280,0.3444,0.3226,1.0,0.30,0,0


In [8]:
differences = {}
for name in sets:
    sample = items[name]
    ids = [item.id for item in sample]
    gold = ex.gold_exact(sample)

    def get(method, name=name, ids=ids):
        return ex.load_predictions(root, name, method, ids)

    differences[name] = {
        "cards minus names": paired_difference(micro_f1, gold, get("funnel_cards"), get("funnel_names")),
        "cards minus trained": paired_difference(micro_f1, gold, get("funnel_cards"), get("trained")),
    }
    if name == ex.EURLEX:
        differences[name]["hybrid minus trained"] = paired_difference(
            micro_f1, gold, get("funnel_hybrid"), get("trained")
        )
        differences[name]["hybrid minus cards"] = paired_difference(
            micro_f1, gold, get("funnel_hybrid"), get("funnel_cards")
        )
within = {
    method: ex.evaluate_within_project(
        humset,
        items[ex.HUMSET],
        ex.load_predictions(root, ex.HUMSET, method, [i.id for i in items[ex.HUMSET]]),
        used,
    )
    for method in results[ex.HUMSET]
}
ex.save_results(
    "04_funnel",
    {
        "results": results,
        "candidate_recall": recall_grid,
        "reached": reached,
        "stages": stages,
        "differences": differences,
        "humset_within_project": within,
        "models": config["models"],
    },
)
differences

{'eurlex': {'cards minus names': (-0.023038249457321774,
   -0.03496096455400584,
   -0.010407648222413843),
  'cards minus trained': (-0.25449416539599634,
   -0.27353096659266596,
   -0.2359926992812019),
  'hybrid minus trained': (-0.17665765047864446,
   -0.19556732581900158,
   -0.15781760785075255),
  'hybrid minus cards': (0.07783651491735191,
   0.06339142746903464,
   0.09131325914254089)},
 'humset': {'cards minus names': (0.04053676487403668,
   0.029160728097932975,
   0.05175625411695008),
  'cards minus trained': (-0.15068075803288283,
   -0.1685344118002129,
   -0.132380498309454)}}